----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

# DesktopDirectSession

This notebook shows how to create, open, monitor, and close a `DesktopDirectSession` with the LSEG Data Library for Python. It follows the `ld.session.desktopdirect` API and the authentication alternatives demonstrated in `test_wag_session.py`.

Run **one** of the session-construction alternatives below, then continue with the common open, verification, and close steps. A Desktop Direct session authenticates through Microsoft Entra ID; it is distinct from a Workspace/Eikon desktop session.

## Prerequisites

- Install a version of `lseg-data` that provides `ld.session.desktopdirect` and use an account/application authorized for Desktop Direct access.
- For the Entra authentication examples, supply the client ID and any authority, scope, or redirect-port values required by your environment. Do not put client secrets or access tokens in this notebook.
- The default-configuration example uses the library configuration selected by `LD_LIB_CONFIG_PATH` or the standard configuration locations.
- The custom-token example requires a callable that obtains and returns a valid access token. The test suite uses `get_entra_token` from its `get_token.py` helper; that helper is not part of this example repository.

In [ ]:
import lseg.data as ld
from datetime import datetime

## Authentication parameters

Replace the client ID with the application ID configured for your Entra setup. These values are used by the explicit-parameter alternatives; omit optional arguments to rely on the API defaults.

In [ ]:
ENTRA_CLIENT_ID = "YOUR_ENTRA_CLIENT_ID"
ENTRA_AUTHORITY = "https://login.microsoftonline.com/organizations"
ENTRA_SCOPE = f"{ENTRA_CLIENT_ID}/.default"
REDIRECT_PORT = 8080

## Option 1 — Specify all Entra authentication parameters

Pass an `EntraAuthentication` provider with the client ID, authority, scope, and redirect port. Use the values registered and approved for your application.

In [ ]:
wag_session = ld.session.desktopdirect.Definition(
    entra_provider=ld.session.desktopdirect.EntraAuthentication(
        entra_client_id=ENTRA_CLIENT_ID,
        entra_authority=ENTRA_AUTHORITY,
        scope=ENTRA_SCOPE,
        redirect_port=REDIRECT_PORT,
    )
).get_session()

## Option 2 — Specify the client ID and authority

The library supplies default values for the scope and redirect port when they are omitted.

In [ ]:
wag_session = ld.session.desktopdirect.Definition(
    entra_provider=ld.session.desktopdirect.EntraAuthentication(
        entra_client_id=ENTRA_CLIENT_ID,
        entra_authority=ENTRA_AUTHORITY,
    )
).get_session()

## Option 3 — Specify only the client ID

The authority and other omitted authentication settings use their library defaults.

In [ ]:
wag_session = ld.session.desktopdirect.Definition(
    entra_provider=ld.session.desktopdirect.EntraAuthentication(
        entra_client_id=ENTRA_CLIENT_ID,
    )
).get_session()

## Option 4 — Use the default library configuration

With no provider argument, the session definition reads the Desktop Direct settings from the library configuration. Ensure that your selected configuration defines the appropriate session and authentication settings before running this cell.

In [ ]:
# Minimal lseg-data.config.json for the default Desktop Direct session.
# The configuration key is hyphenated (entra-client-id), unlike the Python argument.
# {
#   "sessions": {
#     "desktop-direct": {
#       "default": {
#         "entra-client-id": "YOUR_ENTRA_CLIENT_ID"
#       }
#     }
#   }
# }
# The authority, scope, and redirect port use library defaults when omitted.
wag_session = ld.session.desktopdirect.Definition().get_session()

## Option 5 — Supply your own access-token function

Use `EntraProvider` when your application already has a token-acquisition flow. The callback must return a valid access-token string and should handle token-acquisition errors explicitly. In the test project, `get_entra_token` is imported from its local `get_token.py` helper; make that module available on the notebook's Python path before running this example.

In [ ]:
# Example when get_token.py is available on the Python path:
from get_token import get_entra_token

wag_session = ld.session.desktopdirect.Definition(
    entra_provider=ld.session.desktopdirect.EntraProvider(
        get_access_token=get_entra_token
    )
).get_session()

## Observe session state and events

Register callbacks before opening the session to see state transitions and authentication or connection events as they occur.

In [ ]:
def on_state(session, state, message):
    print(f"{datetime.now().time()} - SESSION STATE ({session.name}): {state} | {message}")


def on_event(session, state, message):
    print(f"{datetime.now().time()} - SESSION EVENT ({session.name}): {state} | {message}")


wag_session.on_event(on_event)
wag_session.on_state(on_state)

## Open the session

`get_session()` creates the session object; `open()` starts the connection and authentication. Registering the opened session as the default lets subsequent library calls use it when no session is passed explicitly.

In [ ]:
wag_session.open()
ld.session.set_default(wag_session)
wag_session

## Check the session state

A successful connection should report an open state. If opening fails, inspect the exception and the state/event callback output, then verify your Entra settings and account authorization.

In [ ]:
wag_session.open_state

## Optional: request data through this session

This short historical-pricing request follows the test script and demonstrates that the opened Desktop Direct session can be used for a data request. Choose a RIC and dates available to your account.

In [ ]:
response = ld.content.historical_pricing.summaries.Definition(
    universe="EUR=",
    fields=["BID", "ASK", "OPEN", "HIGH", "LOW", "CLOSE"],
    start=datetime(2026, 5, 1),
    end=datetime(2026, 6, 1),
).get_data(session=wag_session)

response.data.df

## Close the session

Close the session when finished to release its connection and resources.

In [ ]:
wag_session.close()